# Pasajeros (+5): Métricas de Segmentación de Personas y Conteo
Evalúa `yolo11s-seg.pt` (clase `person`) sobre 40–60 imágenes de validación etiquetadas en Roboflow.


In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/peaje/evidencias /content/drive/MyDrive/peaje/modelos
!git clone https://github.com/Adiel05/Control_Peaje.git /content/Control_Peaje 2>/dev/null || git -C /content/Control_Peaje pull
%cd /content/Control_Peaje
!pip -q install -r entrenamiento/requirements.txt


In [ ]:
from google.colab import userdata
from roboflow import Roboflow

WORKSPACE = 'tu-workspace'
PROYECTO = 'peaje-pasajeros-seg'
VERSION = 1

rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
ds = rf.workspace(WORKSPACE).project(PROYECTO).version(VERSION).download('yolov11')
yaml_path = f'{ds.location}/data.yaml'


In [ ]:
# Seccion A: Evaluacion mAP del modelo base COCO (clase person = 0)
import json
from ultralytics import YOLO
modelo = YOLO('yolo11s-seg.pt')
try:
    met = modelo.val(data=yaml_path, classes=[0])
    res_a = {'box_map50': float(met.box.map50), 'seg_map50': float(met.seg.map50), 'seg_map': float(met.seg.map)}
except Exception as e:
    res_a = {'error_val_coco': str(e)}
print('Seccion A:', res_a)


In [ ]:
# Seccion B: Error medio absoluto (MAE) en el conteo de pasajeros por imagen
import glob, os, numpy as np
errores = []
imgs_val = sorted(glob.glob(f'{ds.location}/valid/images/*'))
for img_p in imgs_val:
    stem = os.path.splitext(os.path.basename(img_p))[0]
    lbl_p = f'{ds.location}/valid/labels/{stem}.txt'
    gt = sum(1 for l in open(lbl_p) if l.strip()) if os.path.exists(lbl_p) else 0
    pred = modelo(img_p, classes=[0], verbose=False, conf=0.35)[0]
    pred_cnt = len(pred.boxes) if pred.boxes is not None else 0
    errores.append(abs(gt - pred_cnt))

res_b = {'imagenes_val': len(imgs_val), 'mae_conteo_pasajeros': float(np.mean(errores)) if errores else None}
print('Seccion B:', res_b)
with open('/content/drive/MyDrive/peaje/evidencias/metricas_pasajeros.json', 'w') as f:
    json.dump({**res_a, **res_b}, f, indent=2)


In [ ]:
# Seccion C (opcional): Fine-tuning de 30 epocas si deseas pesos propios
EJECUTAR_FINETUNE = False
if EJECUTAR_FINETUNE:
    import shutil
    modelo.train(data=yaml_path, epochs=30, imgsz=640, batch=16, project='/content/drive/MyDrive/peaje/runs', name='pasajeros_seg')
    shutil.copy('/content/drive/MyDrive/peaje/runs/pasajeros_seg/weights/best.pt', '/content/drive/MyDrive/peaje/modelos/pasajeros_seg.pt')
